In [8]:
import yfinance as yf
import pandas as pd
import os

# 1) Create directory if needed
os.makedirs("../data/processed", exist_ok=True)

# 2) Define tickers and date range
tickers = ['TSLA', 'BND', 'SPY']
start_date = '2015-01-01'
end_date = '2026-01-15'

# 3) Download data from Yahoo Finance
print("Downloading data from Yahoo Finance...")
df_tsla = yf.download('TSLA', start=start_date, end=end_date, progress=False)
df_bnd  = yf.download('BND',  start=start_date, end=end_date, progress=False)
df_spy  = yf.download('SPY',  start=start_date, end=end_date, progress=False)
print("Download complete!")
print(f"TSLA shape: {df_tsla.shape}, BND shape: {df_bnd.shape}, SPY shape: {df_spy.shape}")

# 4) Define a cleaning function
def clean_data_alternative(df, ticker):
    """
    Cleans the DataFrame:
    - Reset index so 'Date' becomes a column
    - Fills numeric missing values via forward then backward fill
    - Adds a 'Ticker' column
    """
    df_clean = df.copy().reset_index()       # Ensure Date is a column
    df_clean['Ticker'] = ticker              # Add ticker label

    # Fill missing numeric values
    for col in df_clean.columns:
        if pd.api.types.is_numeric_dtype(df_clean[col]):
            df_clean[col] = df_clean[col].ffill().bfill()

    return df_clean

# 5) Clean each dataset
print("\nCleaning data...")
df_tsla_clean = clean_data_alternative(df_tsla, 'TSLA')
df_bnd_clean  = clean_data_alternative(df_bnd,  'BND')
df_spy_clean  = clean_data_alternative(df_spy,  'SPY')

print("Samples after cleaning:")
print(df_tsla_clean.head())

# 6) Save cleaned CSV files
print("\nSaving cleaned CSV files...")
df_tsla_clean.to_csv("../data/processed/tsla_cleaned.csv", index=False)
df_bnd_clean.to_csv("../data/processed/bnd_cleaned.csv", index=False)
df_spy_clean.to_csv("../data/processed/spy_cleaned.csv", index=False)
print("Individual cleaned files saved!")

# 7) Save a combined dataset
print("\nCreating combined dataset...")
combined_df = pd.concat([df_tsla_clean, df_bnd_clean, df_spy_clean], ignore_index=True)
combined_df.to_csv("../data/processed/all_tickers_combined.csv", index=False)
print("Combined dataset saved!")

# 8) Optional: Summary of saved files
print("\nVerifying saved files...")
for f in ["tsla_cleaned", "bnd_cleaned", "spy_cleaned", "all_tickers_combined"]:
    path = f"../data/processed/{f}.csv"
    if os.path.exists(path):
        tmp = pd.read_csv(path, nrows=3)
        print(f"{f}.csv — {tmp.shape[0]} rows, {tmp.shape[1]} columns")
    else:
        print(f"{f}.csv not found!")

print("\n" + "="*50)
print("DATA STATISTICS")
print("="*50)

for name, df_clean in [('TSLA', df_tsla_clean),
                       ('BND', df_bnd_clean),
                       ('SPY', df_spy_clean)]:

    print(f"\n{name}:")

    # Confirm Date exists
    if 'Date' in df_clean.columns:
        print(f"Date range: {df_clean['Date'].min()} to {df_clean['Date'].max()}")
    else:
        print("No 'Date' column found!")

    print(f"Rows: {len(df_clean)}")

    # Adj Close stats
    if 'Adj Close' in df_clean.columns:
        # Ensure this is a Series, not DataFrame
        adj_close_series = df_clean['Adj Close']
        if isinstance(adj_close_series, pd.Series):
            min_price = adj_close_series.min()
            max_price = adj_close_series.max()
            print(f"Adj Close min/max: {min_price:.2f}/{max_price:.2f}")
        else:
            print("Adj Close is not a Series — check your columns!")

    # Volume stats
    if 'Volume' in df_clean.columns:
        # ensure this is really a Series
        vol = df_clean['Volume']
        if isinstance(vol, pd.Series):
            try:
                avg_vol = vol.mean()
                print(f"Avg Volume: {avg_vol:,.0f}")
            except Exception as e:
                print("Could not compute average volume:", e)
        else:
            print("Volume column is not a Series — check your data!")






Download complete!
TSLA shape: (2775, 5), BND shape: (2775, 5), SPY shape: (2775, 5)

Cleaning data...
Samples after cleaning:
Price        Date      Close       High        Low       Open    Volume Ticker
Ticker                  TSLA       TSLA       TSLA       TSLA      TSLA       
0      2015-01-02  14.620667  14.883333  14.217333  14.858000  71466000   TSLA
1      2015-01-05  14.006000  14.433333  13.810667  14.303333  80527500   TSLA
2      2015-01-06  14.085333  14.280000  13.614000  14.004000  93928500   TSLA
3      2015-01-07  14.063333  14.318667  13.985333  14.223333  44526000   TSLA
4      2015-01-08  14.041333  14.253333  14.000667  14.187333  51637500   TSLA

Saving cleaned CSV files...
Individual cleaned files saved!

Creating combined dataset...
Combined dataset saved!

Verifying saved files...
tsla_cleaned.csv — 3 rows, 7 columns
bnd_cleaned.csv — 3 rows, 7 columns
spy_cleaned.csv — 3 rows, 7 columns
all_tickers_combined.csv — 3 rows, 17 columns

DATA STATISTICS

TSLA:
